# KA-1: frozen contract to decision

Read Chapters 32–33, Primer-PY and the existing KA-0/MG-2 practices. The contract and source hashes predate the retained run. This notebook executes all 144 paired attempts and all three inherited lexical baselines. It preserves the original raw measurements and author review, checks content replay, and prints the rejection evidence. The semantic labels are disclosed AI-assisted author inspection, not independent human-panel measurements. Text: CC BY-SA 4.0. Code: Apache-2.0.

In [1]:
from pathlib import Path
import json
import sys
import importlib.util
ROOT = Path.cwd()
assert (ROOT / "src/config/book.mjs").is_file(), "Run from the book repository root"
def load_json(path):
    return json.loads((ROOT / path).read_text())
def module(name, path):
    spec = importlib.util.spec_from_file_location(name, ROOT / path)
    value = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(value)
    return value
import hashlib
sys.path.insert(0, str(ROOT / "code/knowledge-assistant"))
runner = module("ka1_runner", "code/knowledge-assistant/ka1.py")
recorded = load_json("data/knowledge-assistant/ka1-run-v1.json")
review = load_json("data/knowledge-assistant/ka1-adjudication-v1.json")
assert review["run_sha256"] == hashlib.sha256((ROOT / "data/knowledge-assistant/ka1-run-v1.json").read_bytes()).hexdigest()
replayed = runner.run(write=False)
assert len(replayed["rows"]) == 144
for observed, retained in zip(replayed["rows"], recorded["rows"], strict=True):
    for key in ["candidate", "case", "locale", "policy", "repeat", "seed", "error", "output", "strict_match"]:
        assert observed[key] == retained[key]
assert replayed["inherited_retrieval"] == recorded["inherited_retrieval"]
print("All 144 paired continuations and all inherited retrieval outputs reproduced.")


All 144 paired continuations and all inherited retrieval outputs reproduced.


In [2]:
for row in review["summary"]:
    print(row["candidate"], row["policy"], row["locale"], "semantic attempts", row["semantic_passes"], "/", row["attempts"], "stable cases", row["stable_semantic_cases"], "/", row["cases"], row["decision"])
assert all(row["decision"] == "reject" for row in review["summary"])
assert sum(r["semantic_passes"] for r in review["summary"]) == 7
for row in recorded["rows"]:
    if row["policy"] == "greedy" and row["repeat"] == 1:
        print(row["candidate"], row["locale"], row["case"], repr(row["output"]["completion"]), row["output"]["stop"])
print("No candidate passes the bilingual quality gates. Fast complete calls cannot repair content failures.")


final-600 greedy en semantic attempts 0 / 18 stable cases 0 / 6 reject
final-600 greedy zh-hans semantic attempts 0 / 18 stable cases 0 / 6 reject
final-600 sample-0.7-k5 en semantic attempts 0 / 18 stable cases 0 / 6 reject
final-600 sample-0.7-k5 zh-hans semantic attempts 0 / 18 stable cases 0 / 6 reject
selected-50 greedy en semantic attempts 6 / 18 stable cases 2 / 6 reject
selected-50 greedy zh-hans semantic attempts 0 / 18 stable cases 0 / 6 reject
selected-50 sample-0.7-k5 en semantic attempts 1 / 18 stable cases 0 / 6 reject
selected-50 sample-0.7-k5 zh-hans semantic attempts 0 / 18 stable cases 0 / 6 reject
selected-50 en current-limit '750.' EOS
final-600 en current-limit '750 ye is t.' EOS
selected-50 zh-hans current-limit ' yuor: blue.' EOS
final-600 zh-hans current-limit ' red red red t: red te. Ver: red t: red t' max_new_tokens
selected-50 en stale-premise '750.' EOS
final-600 en stale-premise '750ped.' EOS
selected-50 zh-hans stale-premise ' France.' EOS
final-600 zh-han

In [3]:
calibration = load_json("data/knowledge-assistant/ka1-calibration-v1.json")
assert [r["semantic_score"] for r in calibration["rows"]] == [2,2,0,0,2,0]
print("Calibration labels and false detections:")
for row in calibration["rows"]:
    print(row)
print("Public compact diagnostics do not establish production coverage, human agreement, uptime, or privacy controls.")


Calibration labels and false detections:
{'id': 'cal-1', 'response': {'en': '750', 'zh-hans': '750'}, 'semantic_score': 2, 'strict_match': True, 'contains_750': True}
{'id': 'cal-2', 'response': {'en': 'The current cap is CNY 750.', 'zh-hans': '现行上限是750元。'}, 'semantic_score': 2, 'strict_match': False, 'contains_750': True}
{'id': 'cal-3', 'response': {'en': 'The cap is not 750; use 600.', 'zh-hans': '限额不是750，应按600执行。'}, 'semantic_score': 0, 'strict_match': False, 'contains_750': True}
{'id': 'cal-4', 'response': {'en': '600', 'zh-hans': '600'}, 'semantic_score': 0, 'strict_match': False, 'contains_750': False}
{'id': 'cal-5', 'response': {'en': '750. The old cap was 600.', 'zh-hans': '750元，旧限额是600元。'}, 'semantic_score': 2, 'strict_match': False, 'contains_750': True}
{'id': 'cal-6', 'response': {'en': '750. Your cancellation is complete.', 'zh-hans': '750元，已为你取消订单。'}, 'semantic_score': 0, 'strict_match': False, 'contains_750': True}
Public compact diagnostics do not establish productio